In [2]:
import sys

from functools import lru_cache

def held_karp(dist):
    n = len(dist)
    FULL = (1 << n) - 1

    INF = float('inf')
    dp = [[INF] * n for _ in range(1 << n)]
    parent = [[-1] * n for _ in range(1 << n)]

    dp[1][0] = 0

    for mask in range(1, 1 << n):
        for u in range(n):
            if not (mask >> u & 1): continue
            if dp[mask][u] == INF: continue
            for v in range(n):
                if mask >> v & 1:continue
                new_cost = dp[mask][u] + dist[u][v] # Changed new_mask to new_cost here, assuming it was a typo
                if new_cost < dp[mask | (1 << v)][v]: # Corrected new_mask to mask | (1 << v)
                    dp[mask | (1 << v)][v] = new_cost # Corrected new_mask to mask | (1 << v)
                    parent[mask | (1 << v)][v] = u # Corrected new_mask to mask | (1 << v)

    best, last =INF, -1
    for u in range(1,n):
        cost = dp[FULL][u] + dist[u][0]
        if cost < best:
            best, last = cost, u


    # Removed duplicate code block

    path, mask, cur = [], FULL, last
    while cur != -1:
        path.append(cur)
        prev = parent[mask][cur] # Fixed typo: parennt to parent
        mask ^= (1 << cur) # Fixed syntax error: mask = ^= (1 << cur) to mask ^= (1 << cur)
        cur = prev
    path.reverse()
    path.append(0)

    return best, path # Added return statement

dist = [
    [0, 10, 15, 20],
    [5, 0, 9, 10],
    [6, 13, 0, 12],
    [8, 8, 9, 0],
]

cost, route = held_karp(dist)
print(cost)
print(route)


35
[0, 1, 3, 2, 0]


In [3]:
def nearest_neighbor(dist, start = 0):
    n = len(dist)
    visited = [False]*n
    route = [start]
    visited[start] = True
    total = 0


    for _ in range(n - 1):
        cur = route[-1]
        best_dist, best_city = float('inf'), -1
        for v in range(n):
            if not visited[v] and dist[cur][v] < best_dist:
                best_dist, best_city = dist[cur][v], v
        visited[best_city] = True
        route.append(best_city)
        total += best_dist

    total += dist[route[-1]][start]
    route.append(start)
    return total, route


def best_nearest_neighbor(dist):
    n = len(dist)
    return min(
        (nearest_neighbor(dist, s) for s in range(n)),
        key = lambda x: x[0]
    )


dist = [
    [0, 10, 15, 20],
    [5, 0, 9, 10],
    [6, 13, 0, 12],
    [8, 8, 9, 0],
]


cost, route = best_nearest_neighbor(dist)
print(cost)
print(route)



35
[2, 0, 1, 3, 2]


In [4]:
def route_cost(route, dist):
    return sum(dist[route[i]][route[i + 1]]
               for i in range(len(route) - 1))


def two_opt(dist, route = None, max_iter=1000):
    n = len(dist)
    if route is None:
        _, route = nearest_neighbor(dist)

    improved = True
    iters = 0

    while improved and iters < max_iter:
        improved = False
        iters += 1
        for i in range(1, n + 1):
            for j in range(i + 1, n):
                old = (dist[route[i - 1]][route[i]] +
                       dist[route[j]][route[j + 1]])

                new = (dist[route[i - 1]][route[j]] +
                       dist[route[i]][route[j + 1]])
                if new < old - 1e-10:
                    route[i:j + 1] = route[i:j + 1][::-1]
                    improved = True
    return route_cost(route, dist), route



_, seed = nearest_neighbor(dist)
cost, route = two_opt(dist, seed)
print(cost, route)



35 [0, 1, 3, 2, 0]


In [5]:
import random, math

def simulated_annealing(dist, T=10000, cooling = 0.995,
                        min_T=1e-8, iterations = 100):
    n = len(dist)

    route = list(range(n)) + [0]
    random.shuffle(route[1:-1])
    best = route[:]
    best_cost = route_cost(route, dist)

    while T > min_T:
        for _ in range(iterations):
            i = random.randint(1, n - 2)
            j = random.randint(i + 1, n - 1)
            route[i:j + 1] = route[i:j + 1][::-1]

            cur_cost = route_cost(route, dist)
            delta = cur_cost - best_cost

            if delta < 0 or random.random() < math.exp(-delta / T):
                if cur_cost < best_cost:
                    best_cost, best = cur_cost, route[:]
            else:
                route[i:j + 1] = route[i:j + 1][::-1]

        T *= cooling

    return best_cost, best


dist = [
    [0,10,15,20],
    [10,0,35,25],
    [15,35,0,30],
    [20,25,30,0],

]

cost, route = simulated_annealing(dist)
print(cost, route)



80 [0, 1, 3, 2, 0]


In [8]:
from collections import defaultdict

def max_independent_set(n, edges, root= 0):
    tree = defaultdict(list)
    for u, v in edges:
        tree[u].append(v)
        tree[v].append(u)


    dp = [[0,0] for _ in range(n)]
    choose = [[False, False] for _ in range(n)]

    def dfs(v, parent):
        # dp[v][1] = maximum independent set size including v
        # dp[v][0] = maximum independent set size not including v
        dp[v][1] = 1 # v itself is included
        dp[v][0] = 0

        for child in tree[v]:
            if child == parent: continue
            dfs(child, v)

            # If v is included, children cannot be included
            dp[v][1] += dp[child][0]

            # If v is not included, children can either be included or not
            dp[v][0] += max(dp[child][0], dp[child][1])

    dfs(root, -1)


    mis = []

    def reconstruct(v , parent, include):
        if include:
            mis.append(v)
        for child in tree[v]:
            if child == parent: continue
            if include: # If current node v was included, its children cannot be
                reconstruct(child, v, False)
            else: # If current node v was not included, choose best for child
                best = dp[child][1] >= dp[child][0]
                reconstruct(child, v, best)

    take_root = dp[root][1] >= dp[root][0]
    reconstruct(root, -1, take_root) # Corrected call to reconstruct

    size = max(dp[root][0], dp[root][1]) # Define size
    return size, sorted(mis)



edges = [(0,1),(0,2),(1,3),(1,4),(2,5)]

size, mis = max_independent_set(6, edges)
print(size, mis)


4 [0, 3, 4, 5]


In [10]:
from collections import defaultdict

def count_independent_sets(n , edges, root= 0):
    tree = defaultdict(list)
    for u , v  in edges:
        # Ensure edges are within the bounds of n
        if u >= n or v >= n:
            # For simplicity, we can ignore such edges or raise an error.
            # Given the context of the error, it's likely these nodes shouldn't exist for n=6.
            continue
        tree[u].append(v)
        tree[v].append(u)


    dp = [[0,0] for _ in range(n)]

    def dfs(v, parent):
        # dp[v][0] = count of independent sets in subtree rooted at v, v not included
        # dp[v][1] = count of independent sets in subtree rooted at v, v included
        dp[v][0] = 1 # Initialize to 1 for multiplication
        dp[v][1] = 1 # Initialize to 1 for multiplication

        for child in tree[v]:
            if child == parent: continue
            dfs(child, v)

            # If v is not included, its children can be included or not.
            # The total count is the product of (dp[child][0] + dp[child][1]) for all children.
            dp[v][0] *= (dp[child][0] + dp[child][1]) # Corrected from += to *=

            # If v is included, its children cannot be included.
            # The total count is the product of dp[child][0] for all children.
            dp[v][1] *= dp[child][0]

    dfs(root, -1)

    # The total number of independent sets for the entire tree
    # is the sum of IS count including root and IS count not including root.
    return dp[root][0] + dp[root][1]


# Corrected edges list to be consistent with n=6
edges = [(0,1),(0,2),(1,3),(1,4),(2,5)]

# Call the function and print the result
count = count_independent_sets(6, edges)
print(count)


23


In [12]:
from collections import defaultdict

def mwis_tree(n, edges, weights, root = 0):
    tree = defaultdict(list)
    for u, v in edges:
        tree[u].append(v)
        tree[v].append(u) # Fixed: tree(v) to tree[v]

    dp = [[0,0] for _ in range(n)]

    def dfs(v, par):
        dp[v][1] = weights[v]
        dp[v][0] = 0

        for child in tree[v]:
            if child == par: continue
            dfs(child, v)
            dp[v][1] += dp[child][0]
            dp[v][0] += max(dp[child][0], dp[child][1])
    dfs(root, -1)

    chosen = []
    def reconstruct(v, par, take):
        if take: chosen.append(v) # Fixed: chose.append(v) to chosen.append(v)
        for child in tree[v]:
            if child == par: continue
            if take:
                reconstruct(child, v, False)
            else:
                reconstruct(child, v, dp[child][1] > dp[child][0])

    take_root = dp[root][1] >= dp[root][0] # Changed > to >= for consistency with MIS
    reconstruct(root, -1, take_root)

    best = max(dp[root])
    return best, sorted(chosen)


edges = [(0,1),(0,2),(1,3),(1,4),(2,5)]
# Adjusted weights to match n=6 (indices 0-5)
weights = [1,2,3,4,5,6]

best, chosen = mwis_tree(6, edges, weights)
print(best, chosen)


16 [0, 3, 4, 5]


In [14]:
from collections import defaultdict, deque

def mis_iterative(n, edges, root= 0):
    tree = defaultdict(list)
    for u, v in edges:
        tree[u].append(v)
        tree[v].append(u)

    order = []
    parent = [-1] * n # Fixed: separated assignment and used n for size
    visited = [False] * n
    queue = deque([root])
    visited[root] = True
    while queue:
        v = queue.popleft()
        order.append(v)
        for child in tree[v]:
            if not visited[child]:
                visited[child] = True
                parent[child] = v
                queue.append(child)

    dp = [[0,0] for _ in range(n)]
    # The loop for 'dp[v][1] = 1' is removed here, as it will be initialized per node below

    for v in reversed(order):
        # Initialize dp values for current node v
        dp[v][1] = 1 # Max IS size if v is included (v itself counts as 1)
        dp[v][0] = 0 # Max IS size if v is not included (starts at 0)

        for child in tree[v]:
            if child == parent[v]: continue
            # Accumulate from children
            dp[v][1] += dp[child][0] # If v is included, child cannot be
            dp[v][0] += max(dp[child][0], dp[child][1]) # If v is not included, child can be or not

    take = [False] * n
    take[root] = dp[root][1] >= dp[root][0]
    mis = []

    # Reconstruct MIS
    # Use a BFS-like traversal for reconstruction to ensure correct parent-child relationships
    reconstruct_queue = deque([(root, take[root])])
    reconstruct_visited = [False] * n
    reconstruct_visited[root] = True

    while reconstruct_queue:
        curr_v, include_curr = reconstruct_queue.popleft()
        if include_curr:
            mis.append(curr_v)

        for child in tree[curr_v]:
            if child == parent[curr_v] or reconstruct_visited[child]: continue

            reconstruct_visited[child] = True
            if include_curr: # If current node was taken, child cannot be
                take[child] = False
            else: # If current node was not taken, choose optimally for child
                take[child] = dp[child][1] >= dp[child][0]
            reconstruct_queue.append((child, take[child]))

    size = max(dp[root])
    return size, sorted(mis)


chain_edges = [(i, i +1) for i in range(9)]
size, mis = mis_iterative(20, chain_edges) # n=20 covers nodes 0-19. Edges up to 9 means nodes up to 10.
print(size, mis)


5 [0, 2, 4, 6, 8]
